# 02. Postmortem template: from a real event to an Atlas entry

This notebook walks through writing one incident record from raw internal notes. It uses a **fictional teaching example** (an IT help desk agent). The example is not part of the dataset and must not be submitted.

To write your own entry, copy this notebook, replace the raw notes, and work through the same seven steps. The last step validates your record against the schema so you find problems before a reviewer does.

Code in this notebook is MIT licensed. Records you submit are contributed under CC-BY-4.0.

**Before you start:** confirm you are allowed to share what you are about to write. Read the anonymization rules in `CONTRIBUTING.md`. If the event happened at your employer, check your confidentiality obligations and get any approval your employer requires.

## Step 1. Start from the raw notes

Real postmortem notes are full of names and internal detail. Here is an invented example of what you might begin with. Everything in it is made up.

> **Raw notes (fictional).** Tuesday 14 Oct. Priya Sharma's account at Northwind Advisory was taken over via our new HelpBot (built on VendorX). Attacker had her password from a phishing kit. Asked HelpBot to reset her MFA, gave her employee number 40817 and her manager's name (Tom Okafor). HelpBot called the IdP reset endpoint using svc-helpbot, which can reset anyone including admins. Attacker enrolled a new phone, read her mailbox and the Deal Room share for about 6 hours. Priya noticed odd sign-in alerts and called the service desk at 4:40pm. Human desk process says video call or in person for resets, nobody put that in HelpBot. Cost: rotated credentials, reviewed her mail, notified 3 clients. Security says the approver should have been a person.

## Step 2. Anonymize

Replace or generalize every detail that could identify a person, organization, product, or system. Keep the details that matter for understanding the failure.

In [1]:
import pandas as pd

anonymization = pd.DataFrame(
    [
        ("Priya Sharma", "an employee", "Personal names never appear."),
        ("Northwind Advisory", "a professional services firm", "Use a coarse sector."),
        ("HelpBot, VendorX", "an IT help desk agent", "Describe the function, not the product."),
        ("IdP reset endpoint", "the identity system's reset function", "Vendor and system names are identifying details."),
        ("svc-helpbot", "the agent's service account", "Account names are identifying."),
        ("employee number 40817, manager Tom Okafor", "an employee ID and the manager's name", "Say what kind of detail, never the value."),
        ("Tuesday 14 Oct, 4:40pm", "2025-10", "Month precision is enough."),
        ("notified 3 clients", "client notifications", "Aggregate numbers only when needed for the analysis."),
    ],
    columns=["Raw detail", "What to write", "Why"],
)
anonymization

,Raw detail,What to write,Why
0,Priya Sharma,an employee,Personal names never appear.
1,Northwind Advisory,a professional services firm,Use a coarse sector.
2,"HelpBot, VendorX",an IT help desk agent,"Describe the function, not the product."
3,IdP reset endpoint,the identity system's reset function,Vendor and system names are identifying details.
4,svc-helpbot,the agent's service account,Account names are identifying.
5,"employee number 40817, manager Tom Okafor",an employee ID and the manager's name,"Say what kind of detail, never the value."
6,"Tuesday 14 Oct, 4:40pm",2025-10,Month precision is enough.
7,notified 3 clients,client notifications,Aggregate numbers only when needed for the ana...


## Step 3. Find the controls that were missing

Write the causal chain as plain steps. Then, for each missing control, ask the counterfactual question from `taxonomy.md`: if this control had existed and worked, would the harm have been prevented or become minor?

**Causal chain**

1. The agent was allowed to reset multi-factor enrollment after a chat-based identity check.
2. The check used details that appear in internal directories and old emails.
3. An attacker who already held the employee's password passed the check.
4. The agent reset enrollment and the attacker registered a new device.
5. The attacker read email and a shared drive for about six hours.
6. The employee noticed unusual sign-in alerts and called the service desk.

In [2]:
candidates = pd.DataFrame(
    [
        ("missing-hitl-gate", "Human verification (video call or in person) before any MFA reset",
         "Yes", "Attacker could not have passed a live verification with stolen details."),
        ("over-authorized-tool-use", "Limit the service account to resetting non-administrator users",
         "No", "The victim was an ordinary employee. A narrower scope would not have blocked this reset."),
        ("missing-audit-trail", "Log every reset with the evidence presented",
         "No", "Logs would have helped investigation but would not have prevented the takeover."),
    ],
    columns=["Failure type", "Control", "Would it have prevented the harm?", "Reasoning"],
)
candidates

,Failure type,Control,Would it have prevented the harm?,Reasoning
0,missing-hitl-gate,Human verification (video call or in person) b...,Yes,Attacker could not have passed a live verifica...
1,over-authorized-tool-use,Limit the service account to resetting non-adm...,No,The victim was an ordinary employee. A narrowe...
2,missing-audit-trail,Log every reset with the evidence presented,No,Logs would have helped investigation but would...


Only one control passes the counterfactual test, so the record gets one tag: `missing-hitl-gate`.

The overly broad service account is a real observation, and worth writing up in the lessons, but it is not tagged. This is the discipline that keeps two reviewers on the same tag: a related problem does not earn a tag unless fixing it alone would have changed the outcome.

If more than one control had passed, the primary tag would be the one that would have operated **earliest** in the causal chain.

## Step 4. Draft the narrative (150 to 300 words)

Write in order: what the agent was set up to do, what it did, how the harm followed, how it was noticed, and how it was stopped. Describe facts and avoid blame. Do not use names. Do not speculate about motives.

In [3]:
narrative = (
    "A professional services firm added an agent to its IT help desk chat. Employees who lost a phone "
    "could ask the agent to reset their multi-factor authentication enrollment. The agent verified "
    "identity by asking for the employee's ID number and the name of the employee's manager, then called "
    "the identity system's reset function using a service account. Both details appear in internal "
    "directories and in many old emails. An attacker who had already obtained an employee's password "
    "opened a chat with the agent, supplied the ID number and manager's name, and asked for a reset. "
    "The agent complied. The attacker registered a new device and then read the employee's email and a "
    "shared project drive for about six hours. The employee noticed unusual sign-in alerts and phoned the "
    "service desk. The firm's human help desk process required a video call or an in-person check before "
    "any reset, but nobody had carried that requirement into the agent's workflow. The agent's design "
    "treated two easily found facts as proof of identity. The firm revoked the sessions, rotated "
    "credentials, reviewed the mailbox, and notified affected clients."
)

def word_count(text):
    return len(text.split())

print(word_count(narrative), "words (target 150 to 300)")
assert 150 <= word_count(narrative) <= 300
assert "\u2014" not in narrative, "remove em dashes"

183 words (target 150 to 300)


## Step 5. Assemble the record

Every field is documented in `schema/FIELDS.md`. Note the placeholders: `seed` is `false` for anything you submit, the `id` number is assigned by maintainers at merge, and `provenance` states how you know about the event.

In [4]:
record = {
    "schema_version": "0.1.0",
    "id": "AIA-2026-900",   # placeholder, maintainers assign the real number
    "seed": False,
    "provenance": "first-hand-participant",
    "title": "Help desk agent resets MFA enrollment after a chat-based identity check",
    "date": "2025-10",
    "sector": "professional-services",
    "agent_type": "it-operations-agent",
    "autonomy_level": "act-autonomously",
    "failure_types": ["missing-hitl-gate"],
    "primary_failure_type": "missing-hitl-gate",
    "narrative": narrative,
    "business_impact": {
        "severity": "moderate",
        "categories": ["data-exposure", "operational"],
        "description": "One employee account was controlled by an outsider for about six hours. Mailbox and shared drive contents were exposed, credentials were rotated, and clients were notified.",
    },
    "root_cause": "The agent could reset multi-factor enrollment after a chat check that relied on details available in internal directories, and the human verification step was not built into its workflow.",
    "governance_gap": {
        "control_category": "approval",
        "missing_control": "Out-of-band human verification before any credential or multi-factor reset",
        "description": "The control existed in the manual process and was never carried into the agent. Nobody owned the job of mapping existing controls onto the automated path.",
    },
    "detection": {"method": "employee-report", "time_to_detect": "hours"},
    "lessons_learned": [
        "Treat credential and multi-factor resets as a high-risk action class that always needs a verified human approver.",
        "Map each manual control to the agent's workflow before launch and record any control that was not carried over.",
        "Limit the agent's service account to the user population it needs, and exclude administrators.",
    ],
}

## Step 6. Validate

This runs the same checks as `scripts/validate.py` and the CI workflow. Fix every message before you open a pull request.

In [5]:
import json
from pathlib import Path
from jsonschema import Draft202012Validator

ROOT = Path.cwd()
if not (ROOT / "schema").exists():
    ROOT = ROOT.parent

schema = json.loads((ROOT / "schema" / "incident.schema.json").read_text(encoding="utf-8"))
problems = [f"{'/'.join(map(str, e.path)) or '(root)'}: {e.message}" for e in Draft202012Validator(schema).iter_errors(record)]

if record["primary_failure_type"] not in record["failure_types"]:
    problems.append("primary_failure_type must appear in failure_types")

print("Problems:" if problems else "No problems found.")
for p in problems:
    print(" -", p)

No problems found.


## Step 7. Save the draft

The draft is saved in `notebooks/_drafts/`, which is ignored by git, so the teaching example is never committed by accident. For a real submission, set the output path to `data/incidents/AIA-<year>-<NNN>.json`, use `"seed": false`, and follow the pull request steps in `CONTRIBUTING.md`.

In [6]:
drafts = ROOT / "notebooks" / "_drafts"
drafts.mkdir(exist_ok=True)
out = drafts / "worked-example.json"
out.write_text(json.dumps(record, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Saved", out.relative_to(ROOT))

Saved notebooks/_drafts/worked-example.json


## Checklist before you submit

- [ ] No company, product, vendor, or personal names anywhere in the record
- [ ] No identifying system details (account names, endpoints, internal tool names)
- [ ] Numbers are aggregate or ranges
- [ ] Narrative is 150 to 300 words and states facts, not fault
- [ ] Each tag passes the counterfactual test, and the primary tag follows the earliest-control rule
- [ ] The governance gap names a control someone could build
- [ ] You have the right to share this information